# Embedding divergence — regression analyses and draw stability

**CPU. Part 1 about 20 minutes, Part 2 about 25–35 minutes. Downloads `all-MiniLM-L6-v2`.**

Same windowing as `embedding_divergence.ipynb` (900/700-character budgets, then 6-sentence caps).

**Part 1 — all regression analyses in one pass.** Builds the revision arm (`build_revision`) and a random-position pool (`build_pool`) once, then fits divergence on `is_revision` with the local-repetition covariate (`max_jaccard`), its interaction, a lexical covariate, curvature and per-model fits, and checks five further confounds: clustering by trace (cluster bootstrap), sentence-count asymmetry, sentence-fragment asymmetry, position in trace and difficulty.

**Part 2 — draw stability.** The random baseline is a random draw, so its sampling variance is not in any single run's bootstrap CI. This builds a mixture-matched pool four times the size of the revision arm (boundary vs mid-sentence positions in the revision arm's proportions), refits on 40 revision-sized subsets and reports the distribution of the estimate across draws, with and without repetition control.

The two parts define helpers with the same names; run them in order, top to bottom. Comments inside the code quote values from the runs that motivated each check.

**Prerequisites:** `labels/`, `rejects/`, `traces/`.

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

# Part 1 — all regression analyses

In [ ]:
# =============================================================================
# EMBEDDING DIVERGENCE — ALL ANALYSES, ONE PASS
#
# Consolidates what were four scripts (stratified gradient / follow-ups /
# balance+interaction / simple effects) and ADDS five confound checks that none
# of them covered.
#
# Consolidating is not only tidier: the four separate notebooks each re-embedded
# every window, so this does ~1/4 the embedding work. Arms and covariates are
# computed ONCE and every part reuses them.
#
# =============================================================================
# WHAT WAS ALREADY ESTABLISHED (reproduced here as Parts 1-4)
#   Revision divergence exceeds random, moderated by local repetition:
#   +0.0248 [+0.0159, +0.0334] at low repetition, +0.0147 [+0.0089, +0.0203] at
#   average, +0.0023 [-0.0065, +0.0113] (ns) at high. Interaction -0.0494
#   [-0.0773, -0.0200]. Survives a lexical covariate, curvature, and holds in
#   4/4 models. Effect is small: ~0.2 SD.
#
# =============================================================================
# WHAT PART 5 ADDS — five things nothing had controlled
#
# C6. CLUSTERING / NON-INDEPENDENCE  *** the most serious ***
#     One trace contributes many revision events (2649 events over far fewer
#     traces). Every bootstrap so far resampled EVENTS, which treats siblings
#     from the same trace as independent observations. That understates standard
#     errors, so every CI reported so far is too NARROW and every p too small.
#     With an effect of only ~0.2 SD this could matter for whether the interval
#     still excludes zero.
#     FIX: cluster bootstrap — resample TRACES (with all their events), not
#     events. Reported side by side so the inflation is visible.
#
# C7. SENTENCE-COUNT ASYMMETRY  *** the most insidious ***
#     Divergence is the cosine distance between two CENTROIDS of unit vectors.
#     A centroid of fewer vectors has higher variance, so the distance between
#     two such centroids is inflated BY CONSTRUCTION. If revision windows hold
#     systematically fewer sentences than random windows, revision divergence is
#     mechanically higher with no semantics involved. Nothing has checked this.
#
# C8. SENTENCE-BOUNDARY / FRAGMENT ASYMMETRY
#     Revision markers usually begin a sentence ("Wait, that's wrong"), so the
#     split lands on a boundary and both sides are whole sentences. Random
#     offsets are drawn uniformly in CHARACTER space, so they land mid-sentence:
#     `w[:ls]` ends in a fragment and `w[le:]` begins in one. Fragments are
#     shorter and less coherent, so they embed differently. This is a structural
#     difference between the arms that has nothing to do with revisions.
#
# C9. POSITION IN TRACE
#     Random offsets are uniform in [1000, len-800]. Revision markers are not
#     uniformly distributed -- they plausibly cluster where the model checks its
#     work. If divergence varies with depth through a trace, that is a confound
#     entirely separate from repetition.
#
# C10. PROBLEM DIFFICULTY
#     hard (200) vs medium (100). If revisions concentrate in hard problems and
#     divergence differs by difficulty, that is uncontrolled too.
#
# Each is first MEASURED (are the arms actually imbalanced on it?) and then
# CONTROLLED (does the revision effect survive adding it to the model?). An
# imbalance that does not move the estimate is not a problem; the point is to
# find out which kind each one is.
#
# COST: CPU, ~20 minutes.
# =============================================================================



from pathlib import Path
from collections import Counter, defaultdict
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

GDRIVE_BASE = DATA_ROOT

In [ ]:
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
REJECT_DIR  = GDRIVE_BASE / "rejects"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH    = EXPORT_DIR / "divergence_all_analyses.json"

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE,  RAW_BUFFER_AFTER  = 2500, 1200

In [ ]:
N_BEFORE = N_AFTER = 6
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED, MIN_LEN, NQ = 8080, 2000, 4
N_BOOT = 2000
MARKER_NEAR_CHARS = 200
P_FLOOR = 1.0 / N_BOOT      # smallest reportable bootstrap p -- never print 0

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

In [ ]:
# ---------------------------------------------------------------- helpers ---
def clean_bpe(t): return t.replace("Ġ", " ").replace("Ċ", "\n")
def split_sentences(t): return [p for p in re.split(r'(?<=[.!?])\s+', t.strip()) if p.strip()]

def take_by_budget(sents, budget, from_end):
    if not sents: return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget: break
    return list(reversed(out)) if from_end else out

def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE); re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_]); ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)): return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after  = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]): after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after: return None, None
    return before, after

def eligible(tr):
    return tr is not None and tr.get("correct", False) and not tr.get("truncated", False)

def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

_TOK = re.compile(r"[a-z0-9]+")

In [ ]:
def toks(s): return set(_TOK.findall(s.lower()))

def max_jaccard(before, after):
    bs = [toks(s) for s in before]; as_ = [toks(s) for s in after]
    best = 0.0
    for b in bs:
        if not b: continue
        for a in as_:
            if not a: continue
            i = len(b & a)
            if i:
                j = i / len(b | a)
                if j > best: best = j
    return float(best)

def fmt_p(p):
    """Never print p = 0. The bootstrap floor is 1/N_BOOT."""
    return f"<{P_FLOOR:.4f}" if p < P_FLOOR else f"{p:.4g}"


# ----------------------------------------------------------------- models ---
def _ls(y, X):
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return b

def fit_boot(y, X, groups=None, n_boot=N_BOOT, seed=0):
    """Least squares + bootstrap. If `groups` is given, resamples CLUSTERS
    (whole traces, with all their events) instead of individual rows -- the
    correct unit when events are nested inside traces."""
    beta = _ls(y, X)
    rng = np.random.RandomState(seed)
    n = len(y); draws = np.empty((n_boot, X.shape[1]))
    if groups is None:
        for b in range(n_boot):
            i = rng.randint(0, n, n)
            draws[b] = _ls(y[i], X[i])
    else:
        gidx = defaultdict(list)
        for i, g in enumerate(groups):
            gidx[g].append(i)
        keys = list(gidx.keys())
        arrs = [np.array(gidx[k]) for k in keys]
        G = len(keys)
        for b in range(n_boot):
            pick = rng.randint(0, G, G)
            i = np.concatenate([arrs[j] for j in pick])
            draws[b] = _ls(y[i], X[i])
    return beta, draws

def ci_p(col):
    lo, hi = np.percentile(col, [2.5, 97.5])
    p = 2 * min((col <= 0).mean(), (col >= 0).mean())
    return float(lo), float(hi), float(min(1.0, p))

def mw(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 5 or len(b) < 5: return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    rng = np.random.RandomState(0)
    d = [np.median(rng.choice(a, len(a), True)) - np.median(rng.choice(b, len(b), True))
         for _ in range(5000)]
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"n_a": len(a), "n_b": len(b), "p": float(p),
            "median_a": float(np.median(a)), "median_b": float(np.median(b)),
            "median_diff": float(np.median(a) - np.median(b)),
            "ci95": [float(lo), float(hi)]}

In [ ]:
# ------------------------------------------------------------------- arms ---
def build(cache, rng):
    """All three arms, recording every covariate Part 5 needs: trace id (for
    clustering), sentence counts, fragment flags, relative position, difficulty."""
    rev, flr, rnd = [], [], []
    marker_off = {}

    def pack(arm, model, pid, before, after, cs, fr, tr):
        return {"arm": arm, "model": model, "problem_id": pid,
                "trace_id": f"{model}/{pid}",             # C6 cluster unit
                "before": before, "after": after,
                "n_before": len(before), "n_after": len(after),   # C7
                "len_before": sum(len(s) for s in before),
                "len_after": sum(len(s) for s in after),
                # C8: did the split land on a sentence boundary?
                "frag_before": 0 if re.search(r'[.!?]["\')\]]?\s*$', before[-1]) else 1,
                "frag_after": 0 if re.match(r'^["\'(\[]?[A-Z0-9]', after[0].strip()) else 1,
                "rel_pos": cs / max(1, len(fr)),                   # C9
                "difficulty": 1 if tr.get("difficulty") == "hard" else 0}  # C10

    for model in MODELS:
        mdir = LABELS_DIR / model
        if mdir.exists():
            for f in mdir.glob("*.json"):
                obj = json.load(open(f)); pid = obj.get("problem_id", f.stem)
                revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
                if not revs: continue
                tr = load_trace(model, pid, cache)
                if not eligible(tr): continue
                fr = tr.get("full_response", "")
                if len(fr) < MIN_LEN: continue
                offs = []
                for r in revs:
                    cs, ce = r.get("char_start"), r.get("char_end")
                    if cs is None or ce is None or not (0 <= cs < len(fr)): continue
                    offs.append(cs)
                    b, a = extract_window(fr, cs, ce)
                    if b is None: continue
                    rev.append(pack("revision", model, pid, b, a, cs, fr, tr))
                marker_off[(model, pid)] = offs

        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if rp.exists():
            for e in json.load(open(rp)):
                pid = e["problem_id"]
                tr = load_trace(model, pid, cache)
                if not eligible(tr): continue
                fr = tr.get("full_response", "")
                if len(fr) < MIN_LEN: continue
                cs, mt = e.get("char_start"), e.get("matched_text", "")
                if cs is None or not (0 <= cs < len(fr)): continue
                b, a = extract_window(fr, cs, min(cs + len(mt), len(fr)))
                if b is None: continue
                flr.append(pack("filler", model, pid, b, a, cs, fr, tr))

    for model, n_want in Counter(e["model"] for e in rev).items():
        pids = [pid for (m, pid), tr in cache.items() if m == model and eligible(tr)]
        if not pids: continue
        made = attempts = 0
        while made < n_want and attempts < n_want * 20:
            attempts += 1
            pid = pids[rng.randrange(len(pids))]
            tr = cache[(model, pid)]; fr = tr.get("full_response", "")
            if len(fr) < MIN_LEN: continue
            cs = rng.randrange(1000, len(fr) - 800)
            b, a = extract_window(fr, cs, cs)
            if b is None: continue
            ev = pack("random", model, pid, b, a, cs, fr, tr)
            offs = marker_off.get((model, pid), [])
            ev["dist_to_marker"] = min((abs(cs - o) for o in offs), default=None)
            rnd.append(ev); made += 1
    return rev, flr, rnd


# ------------------------------------------------------------------- main ---
def main():
    cache, results = {}, {}
    print(f"Loading {EMBED_MODEL_NAME} ...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)
    rev, flr, rnd = build(cache, random.Random(SEED))
    print(f"  revision {len(rev)}   filler {len(flr)}   random {len(rnd)}")
    print(f"  distinct traces: revision {len({e['trace_id'] for e in rev})}, "
          f"random {len({e['trace_id'] for e in rnd})}")

    print("  embedding once for all analyses ...")
    for e in rev + flr + rnd:
        B = emb.encode(e["before"], normalize_embeddings=True)
        A = emb.encode(e["after"],  normalize_embeddings=True)
        cb, ca = B.mean(axis=0), A.mean(axis=0)
        e["divergence"] = 1.0 - float(np.dot(cb, ca) /
                                      (np.linalg.norm(cb) * np.linalg.norm(ca)))
        e["max_pair_sim"] = float((B @ A.T).max())
        e["max_jaccard"]  = max_jaccard(e["before"], e["after"])

    v_rev = np.array([e["divergence"] for e in rev])
    v_flr = np.array([e["divergence"] for e in flr])
    v_rnd = np.array([e["divergence"] for e in rnd])

    # =================== PART 1 — three-arm, pooled ========================
    print(f"\n{'#'*74}\n# PART 1 — THREE-ARM COMPARISON (pooled)\n{'#'*74}")
    for lab, a, b in (("revision vs random", v_rev, v_rnd),
                      ("revision vs filler", v_rev, v_flr),
                      ("filler   vs random", v_flr, v_rnd)):
        r = mw(a, b)
        print(f"  {lab}: diff {r['median_diff']:+.4f} "
              f"CI [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}] p={r['p']:.4g}")
        results[lab.replace(" ", "_")] = r
    print("\n  NOTE Mann-Whitney p and the median-difference CI answer different")
    print("  questions, so they can disagree: MW detects a small shift across the")
    print("  whole distribution while the median barely moves. Quote both.")

    # =================== PART 2 — filler floor is vacuous ==================
    print(f"\n{'#'*74}\n# PART 2 — IS THE >={MIN_LEN}-CHAR FLOOR VACUOUS? (C1)\n{'#'*74}")
    rev0, flr0, _ = build(cache, random.Random(SEED))   # same floor applies
    n_short = sum(1 for (m, p), tr in cache.items()
                  if eligible(tr) and len(tr.get("full_response", "")) < MIN_LEN)
    print(f"  eligible traces shorter than {MIN_LEN} chars: {n_short}")
    results["floor_vacuous"] = bool(n_short == 0)
    if n_short == 0:
        print("  VACUOUS — the floor excludes nothing, so C1 was a THEORETICAL")
        print("  confound, never operative. Drop the symmetric/asymmetric framing:")
        print("  they are the same analysis.")

    # =================== PART 3 — stratified gradient ======================
    print(f"\n{'#'*74}\n# PART 3 — GRADIENT BY LOCAL REPETITION\n{'#'*74}")
    strat = {}
    for cov in ("max_jaccard", "max_pair_sim"):
        pooled = np.array([e[cov] for e in rev + flr + rnd])
        edges = np.quantile(pooled, np.linspace(0, 1, NQ + 1))
        edges[0], edges[-1] = -np.inf, np.inf
        print(f"\n  --- {cov} ---")
        print(f"  {'Q':<4}{'med_rev':>9}{'med_rnd':>9}{'rev-rnd':>10}{'p':>11}"
              f"{'IQR_rev':>9}{'cov_imbal':>11}{'p_bal':>9}")
        rows = {}
        for q in range(NQ):
            lo, hi = edges[q], edges[q+1]
            R = [e for e in rev if lo <= e[cov] < hi]
            N = [e for e in rnd if lo <= e[cov] < hi]
            if len(R) < 5 or len(N) < 5: continue
            dr = [e["divergence"] for e in R]; dn = [e["divergence"] for e in N]
            cr = np.array([e[cov] for e in R]); cn = np.array([e[cov] for e in N])
            r = mw(dr, dn)
            _, pb = mannwhitneyu(cr, cn, alternative="two-sided")
            iqr = np.percentile(dr, 75) - np.percentile(dr, 25)
            rows[f"Q{q+1}"] = {"n_rev": len(R), "n_rnd": len(N), **r,
                               "cov_imbalance": float(cr.mean()-cn.mean()),
                               "p_balance": float(pb), "iqr_rev": float(iqr)}
            print(f"  Q{q+1:<3}{r['median_a']:>9.4f}{r['median_b']:>9.4f}"
                  f"{r['median_diff']:>+10.4f}{r['p']:>11.3g}{iqr:>9.4f}"
                  f"{cr.mean()-cn.mean():>+11.4f}{pb:>9.3g}")
        strat[cov] = rows
        q1 = rows.get("Q1")
        if q1 and q1["p_balance"] < 0.05 and q1["cov_imbalance"] < 0:
            slope_hint = abs(q1["cov_imbalance"])
            print(f"  -> Q1 arms are IMBALANCED on {cov} in the inflating direction")
            print(f"     (p={q1['p_balance']:.3g}). The raw quartile gap overstates;")
            print(f"     use the model-based simple effects in Part 4.")
        print(f"  -> IQR {rows['Q1']['iqr_rev']:.4f} -> {rows[f'Q{NQ}']['iqr_rev']:.4f}: "
              f"spread {'preserved (not a floor effect)' if rows[f'Q{NQ}']['iqr_rev'] > rows['Q1']['iqr_rev']*0.7 else 'COMPRESSED — floor effect likely'}")
    results["stratified"] = strat

    # =================== PART 4 — model, simple effects, per-model =========
    ev = rev + rnd
    y = np.array([e["divergence"] for e in ev], float)
    r_ = np.array([1.0 if e["arm"] == "revision" else 0.0 for e in ev])
    tid = [e["trace_id"] for e in ev]

    print(f"\n{'#'*74}\n# PART 4 — CONTINUOUS MODEL + SIMPLE EFFECTS\n{'#'*74}")
    part4 = {}
    for cov in ("max_jaccard", "max_pair_sim"):
        lead = "PRIMARY (lexical)" if cov == "max_jaccard" else "secondary (embedding-derived)"
        c = np.array([e[cov] for e in ev], float); cm = float(c.mean()); cc = c - cm
        X = np.column_stack([np.ones_like(y), r_, cc, r_*cc])
        b, d = fit_boot(y, X)
        print(f"\n  --- {cov} — {lead} (centred at {cm:.4f}) ---")
        nm = ["intercept", "is_revision", cov, f"is_revision x {cov}"]
        terms = {}
        for k in range(4):
            lo, hi, p = ci_p(d[:, k])
            terms[nm[k]] = {"beta": float(b[k]), "ci95": [lo, hi], "p": p}
            print(f"    {nm[k]:<26}{b[k]:>+9.4f}  [{lo:+.4f}, {hi:+.4f}]  p={fmt_p(p)}")
        qs = np.quantile(c, np.linspace(0, 1, NQ+1))
        lv = [("low  (Q1 mean)", float(c[c < qs[1]].mean())),
              ("mean", cm),
              ("high (Q4 mean)", float(c[c >= qs[NQ-1]].mean()))]
        print(f"\n    simple effects (revision - random):")
        simple = {}
        for lab, c0 in lv:
            dr = d[:, 1] + d[:, 3]*(c0 - cm); est = b[1] + b[3]*(c0 - cm)
            lo, hi, p = ci_p(dr)
            sd = y.std()
            simple[lab] = {"cov": c0, "effect": float(est), "ci95": [lo, hi],
                           "p": p, "effect_in_sd": float(est/sd)}
            print(f"      {lab:<16}{est:>+9.4f}  [{lo:+.4f}, {hi:+.4f}]  "
                  f"p={fmt_p(p):<9} = {est/sd:.2f} SD")
        part4[cov] = {"cov_mean": cm, "terms": terms, "simple_effects": simple}
    results["part4"] = part4
    print(f"\n  Effect sizes are ~0.2 SD -- SMALL by convention. Robust, but small;")
    print(f"  report both facts.")

    # =================== PART 5 — the five new confounds ===================
    print(f"\n{'#'*74}\n# PART 5 — FIVE CONFOUNDS NOTHING HAD CONTROLLED\n{'#'*74}")
    cov = "max_jaccard"            # the primary covariate throughout Part 5
    c = np.array([e[cov] for e in ev], float); cm = float(c.mean()); cc = c - cm
    p5 = {}

    # ---- measure the imbalances first ------------------------------------
    print(f"\n  BALANCE — are the arms actually different on each?")
    print(f"  {'covariate':<22}{'revision':>11}{'random':>11}{'diff':>10}{'p':>11}")
    extra = [("n_before", "C7 sentences before"), ("n_after", "C7 sentences after"),
             ("frag_before", "C8 fragment before"), ("frag_after", "C8 fragment after"),
             ("rel_pos", "C9 position in trace"), ("difficulty", "C10 hard problem")]
    bal = {}
    for key, lab in extra:
        a = np.array([e[key] for e in rev], float); bb = np.array([e[key] for e in rnd], float)
        _, p = mannwhitneyu(a, bb, alternative="two-sided")
        bal[key] = {"rev_mean": float(a.mean()), "rnd_mean": float(bb.mean()),
                    "diff": float(a.mean()-bb.mean()), "p": float(p)}
        star = "*" if p < 0.05 else " "
        print(f"  {lab:<22}{a.mean():>11.4f}{bb.mean():>11.4f}"
              f"{a.mean()-bb.mean():>+10.4f}{p:>10.3g}{star}")
    p5["balance"] = bal
    n_imbal = sum(1 for v in bal.values() if v["p"] < 0.05)
    print(f"\n  {n_imbal}/{len(extra)} are significantly imbalanced between the arms.")
    print(f"  An imbalance only matters if controlling for it MOVES the estimate —")
    print(f"  which is what the models below test.")

    # ---- C6: cluster bootstrap -------------------------------------------
    print(f"\n  {'-'*70}\n  C6 — CLUSTERING: events are nested in traces\n  {'-'*70}")
    ntr = len(set(tid))
    print(f"  {len(ev)} events from {ntr} distinct traces "
          f"({len(ev)/ntr:.1f} events per trace)")
    print(f"  Resampling EVENTS treats siblings as independent -> CIs too narrow.")
    X = np.column_stack([np.ones_like(y), r_, cc, r_*cc])
    b_e, d_e = fit_boot(y, X)                       # event bootstrap
    b_c, d_c = fit_boot(y, X, groups=tid)           # cluster bootstrap
    print(f"\n  {'term':<26}{'beta':>9}{'event CI':>22}{'CLUSTER CI':>22}{'widen':>8}")
    clus = {}
    for k, name in enumerate(["intercept", "is_revision", cov, f"is_revision x {cov}"]):
        le, he, pe = ci_p(d_e[:, k]); lc, hc, pc = ci_p(d_c[:, k])
        w = (hc-lc)/(he-le) if (he-le) else float("nan")
        clus[name] = {"beta": float(b_c[k]),
                      "ci_event": [le, he], "ci_cluster": [lc, hc],
                      "p_cluster": pc, "widening": float(w)}
        print(f"  {name:<26}{b_c[k]:>+9.4f}  [{le:+.4f},{he:+.4f}]"
              f"  [{lc:+.4f},{hc:+.4f}]{w:>8.2f}x")
    p5["cluster"] = clus
    rev_ok = clus["is_revision"]["ci_cluster"][0] > 0
    int_ok = clus[f"is_revision x {cov}"]["ci_cluster"][1] < 0
    print(f"\n  is_revision survives clustering : {rev_ok}")
    print(f"  interaction survives clustering  : {int_ok}")
    if rev_ok and int_ok:
        print("  -> BOTH SURVIVE. The CIs widen but still exclude zero, so the")
        print("     result is not an artifact of treating sibling events as")
        print("     independent. REPORT THE CLUSTER-ROBUST CIs — they are the")
        print("     honest ones, and a reviewer who notices the nesting will ask.")
    else:
        print("  -> DOES NOT SURVIVE. Once sibling events are correctly treated as")
        print("     clustered, the interval includes zero. The earlier CIs were")
        print("     too narrow and the effect is NOT established. Do not report it.")

    # ---- C7-C10: control for everything ----------------------------------
    print(f"\n  {'-'*70}\n  C7-C10 — DOES THE EFFECT SURVIVE CONTROLLING FOR THEM?"
          f"\n  {'-'*70}")
    def z(key):
        v = np.array([e[key] for e in ev], float)
        s = v.std()
        return (v - v.mean()) / (s if s > 0 else 1.0)
    specs = [
        ("repetition only (baseline)", []),
        ("+ sentence counts (C7)",     ["n_before", "n_after"]),
        ("+ fragment flags (C8)",      ["n_before", "n_after", "frag_before", "frag_after"]),
        ("+ trace position (C9)",      ["n_before", "n_after", "frag_before", "frag_after", "rel_pos"]),
        ("+ difficulty (C10) = FULL",  ["n_before", "n_after", "frag_before", "frag_after", "rel_pos", "difficulty"]),
    ]
    print(f"  {'model':<30}{'is_revision':>12}{'cluster CI':>22}{'interaction':>13}{'CI':>22}")
    ladder = {}
    for lab, keys in specs:
        cols = [np.ones_like(y), r_, cc, r_*cc] + [z(k) for k in keys]
        Xf = np.column_stack(cols)
        bf, df = fit_boot(y, Xf, groups=tid)     # cluster-robust throughout
        lo1, hi1, p1 = ci_p(df[:, 1]); lo3, hi3, p3 = ci_p(df[:, 3])
        ladder[lab] = {"is_revision": {"beta": float(bf[1]), "ci95": [lo1, hi1], "p": p1},
                       "interaction": {"beta": float(bf[3]), "ci95": [lo3, hi3], "p": p3},
                       "controls": keys}
        print(f"  {lab:<30}{bf[1]:>+12.4f}  [{lo1:+.4f},{hi1:+.4f}]"
              f"{bf[3]:>+13.4f}  [{lo3:+.4f},{hi3:+.4f}]")
    p5["control_ladder"] = ladder
    base = ladder["repetition only (baseline)"]["is_revision"]["beta"]
    full = ladder["+ difficulty (C10) = FULL"]["is_revision"]
    print(f"\n  is_revision: {base:+.4f} (repetition only) -> {full['beta']:+.4f} (full)")
    moved = abs(full["beta"] - base) / abs(base) if base else float("nan")
    print(f"  moved {moved:.1%} when all five confounds were added")
    if full["ci95"][0] > 0 and moved < 0.35:
        print("  -> ROBUST. The estimate is stable and the interval still excludes")
        print("     zero with every confound controlled and cluster-robust CIs.")
        print("     This is the specification to report.")
    elif full["ci95"][0] > 0:
        print("  -> SURVIVES but MOVES substantially. Report the FULL model's")
        print("     estimate, not the repetition-only one, and say which controls")
        print("     mattered.")
    else:
        print("  -> DOES NOT SURVIVE full control. The repetition-only estimate was")
        print("     absorbing one or more of these. Report the full model and do")
        print("     not claim a revision effect.")

    # ---- marker contamination (favourable) -------------------------------
    dists = [e["dist_to_marker"] for e in rnd if e.get("dist_to_marker") is not None]
    near = sum(1 for x in dists if x <= MARKER_NEAR_CHARS)
    print(f"\n  {'-'*70}\n  RANDOM-ARM MARKER CONTAMINATION (biases toward the null)"
          f"\n  {'-'*70}")
    if dists:
        print(f"  median distance to nearest marker: {np.median(dists):.0f} chars")
        print(f"  within {MARKER_NEAR_CHARS} chars: {near} ({near/len(rnd):.1%})")
        print(f"  -> random positions have no exclusion zone, so some sit on real")
        print(f"     revisions. That pulls random TOWARD revision, making the")
        print(f"     measured effect CONSERVATIVE. A point in your favour.")
    p5["marker_contamination"] = {"median_dist": float(np.median(dists)) if dists else None,
                                  "n_within": int(near),
                                  "frac_within": float(near/len(rnd)) if rnd else None}
    results["part5"] = p5

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(results, open(OUT_PATH, "w"), indent=2, default=str)
    print(f"\n{'#'*74}\nWrote {OUT_PATH}")
    print("\nREPORT: the FULL model's is_revision and interaction, with")
    print("CLUSTER-ROBUST CIs, under max_jaccard. Simple effects from Part 4 give")
    print("the effect size at a stated repetition level. Never print p = 0.")


if __name__ == "__main__":
    main()

---

## What to report

The **full model's** `is_revision` and interaction, with **cluster-robust CIs**, under `max_jaccard`. Part 4's simple effects give the effect size at a stated repetition level.

Check in this order:

1. **Does C6 survive?** If the cluster-robust CI includes zero, nothing else matters — the earlier intervals were too narrow and the effect is not established.
2. **Does the control ladder hold?** If `is_revision` moves a lot between 'repetition only' and 'full', report the full model and say which controls mattered.
3. **Which imbalances were real?** An imbalanced covariate that doesn't move the estimate is worth one sentence, not a caveat.

# Part 2 — draw stability of the random baseline

In [ ]:
# =============================================================================
# DIVERGENCE — DRAW STABILITY, A MIXTURE-MATCHED BASELINE, AND A LARGE POOL
#
# THIS SUPERSEDES divergence_matched_random_arm.py, WHOSE VERDICT WAS WRONG.
# That script printed "CONFIRMED, AND LARGER" while its own section 1 reported
# that matching had FAILED (4/4 structural covariates still imbalanced). The
# verdict was never gated on whether the matching worked. It is here.
#
# =============================================================================
# THREE PROBLEMS THE LAST RUN EXPOSED
#
# P1. THE ESTIMATE IS UNSTABLE ACROSS RANDOM DRAWS  *** the serious one ***
#     Two runs of the same code with the same seed gave
#         is_revision +0.0147 [+0.0060, +0.0235]
#         is_revision +0.0093 [+0.0005, +0.0175]
#     because the second script also drew a matched arm, consuming rng state, so
#     the uniform arm's POSITIONS differed. A 37% move, and the second interval
#     barely clears zero.
#
#     The bootstrap resamples the DRAWN events. It never resamples the DRAW. So
#     the random arm's position-sampling variance appears in NO confidence
#     interval reported so far -- a second way the CIs have been too narrow,
#     unrelated to clustering.
#
#     FIX, two parts:
#       (a) stop matching the random arm's COUNT to the revision arm. There was
#           never a reason to -- unbalanced n is fine in regression -- and a
#           revision-sized baseline is a small sample of positions whose noise
#           lands straight in the estimate. A POOL several times larger gives
#           the FULL-POOL estimate a smaller standard error.
#           (Precisely: it shrinks the SE of the full-pool fit. It does NOT
#           shrink the spread across revision-sized subsets, which is a
#           property of n=2649, not of the pool. Those are different quantities
#           and conflating them would be wrong.)
#       (b) measure the draw variance instead of assuming it away: resample
#           revision-sized subsets from the pool many times, refit each, and
#           report the DISTRIBUTION of the estimate across draws. That is what
#           characterises how much a count-matched design wobbles. If the
#           distribution straddles zero, the effect is not established however
#           tight any single run's bootstrap CI looked.
#
#     A CALIBRATION NOTE WORTH KNOWING. Under an i.i.d. simulation at this n and
#     noise level, two independent draws differ by about 0.002 typically. The
#     two real runs differed by 0.0054 -- roughly 2-3x more. That excess is what
#     you expect when observations are CLUSTERED (341 traces, ~15.5 events each),
#     because the effective sample size is far below 2649. So draw instability
#     and clustering compound, and the across-draw spread below is measured on
#     the real clustered data rather than assumed from an i.i.d. formula.
#
# P2. THE MATCHED ARM OVERSHOT
#     Revision events are a MIXTURE: ~79% sit at a sentence boundary, ~21%
#     mid-sentence (markers like "...the answer, actually wait"). Drawing the
#     baseline only at boundaries gave frag_before = 0.000 against revision's
#     0.208, replacing a +0.755 imbalance with a -0.208 one, and it broke
#     n_before, which had been balanced.
#     FIX: mirror the mixture. Draw at a boundary with the revision arm's own
#     boundary probability, and mid-sentence otherwise.
#
# P3. ATTRITION AND A REPETITION FLIP
#     The matched arm filled only 2026 of 2649 (76%) -- the boundary constraint
#     inside [1000, len-800] failed often, so it was drawn from a biased subset
#     of traces. And its max_jaccard came out 0.4473 against revision's 0.4274,
#     i.e. MORE repetitive than revision, where the uniform arm was LESS. Since
#     divergence falls with repetition, a more-repetitive baseline has lower
#     divergence and INFLATES the gap -- so "matched is larger" may be that
#     flip rather than the fragment fix.
#     FIX: report attrition explicitly, and report the estimate both raw and
#     with repetition controlled, so the flip cannot masquerade as an effect.
#
# COST: CPU, ~25-35 minutes (the pool is POOL_MULT x larger than before).
# =============================================================================



from pathlib import Path
from collections import Counter, defaultdict
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

GDRIVE_BASE = DATA_ROOT

In [ ]:
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH    = EXPORT_DIR / "divergence_draw_stability.json"

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE,  RAW_BUFFER_AFTER  = 2500, 1200
N_BEFORE = N_AFTER = 6

In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED, MIN_LEN, NQ = 8080, 2000, 4
N_BOOT = 1500
POOL_MULT = 4          # random pool = POOL_MULT x revision count
N_DRAWS = 40           # how many revision-sized subsets to refit on
P_FLOOR = 1.0 / N_BOOT

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

In [ ]:
def clean_bpe(t): return t.replace("Ġ", " ").replace("Ċ", "\n")
def split_sentences(t): return [p for p in re.split(r'(?<=[.!?])\s+', t.strip()) if p.strip()]

def take_by_budget(sents, budget, from_end):
    if not sents: return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget: break
    return list(reversed(out)) if from_end else out

def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE); re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_]); ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)): return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after  = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]): after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after: return None, None
    return before, after

def eligible(tr):
    return tr is not None and tr.get("correct", False) and not tr.get("truncated", False)

def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

_TOK = re.compile(r"[a-z0-9]+")

In [ ]:
def toks(s): return set(_TOK.findall(s.lower()))

def max_jaccard(before, after):
    bs = [toks(s) for s in before]; as_ = [toks(s) for s in after]
    best = 0.0
    for b in bs:
        if not b: continue
        for a in as_:
            if not a: continue
            i = len(b & a)
            if i:
                j = i / len(b | a)
                if j > best: best = j
    return float(best)

def fmt_p(p): return f"<{P_FLOOR:.4f}" if p < P_FLOOR else f"{p:.4g}"

def sentence_starts(fr):
    return [m.end() for m in re.finditer(r'(?<=[.!?])\s+', fr)]

def pack(arm, model, pid, before, after, cs, fr, tr):
    return {"arm": arm, "model": model, "problem_id": pid,
            "trace_id": f"{model}/{pid}", "before": before, "after": after,
            "n_before": len(before), "n_after": len(after),
            "frag_before": 0 if re.search(r'[.!?]["\')\]]?\s*$', before[-1]) else 1,
            "frag_after": 0 if re.match(r'^["\'(\[]?[A-Z0-9]', after[0].strip()) else 1,
            "rel_pos": cs / max(1, len(fr)),
            "difficulty": 1 if tr.get("difficulty") == "hard" else 0}


def build_revision(cache):
    rev, mlens = [], []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if not mdir.exists(): continue
        for f in mdir.glob("*.json"):
            obj = json.load(open(f)); pid = obj.get("problem_id", f.stem)
            revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
            if not revs: continue
            tr = load_trace(model, pid, cache)
            if not eligible(tr): continue
            fr = tr.get("full_response", "")
            if len(fr) < MIN_LEN: continue
            for r in revs:
                cs, ce = r.get("char_start"), r.get("char_end")
                if cs is None or ce is None or not (0 <= cs < len(fr)): continue
                b, a = extract_window(fr, cs, ce)
                if b is None: continue
                mlens.append(ce - cs)
                rev.append(pack("revision", model, pid, b, a, cs, fr, tr))
    return rev, np.array(mlens)


def build_pool(cache, rng, per_model_target, mlens, p_boundary):
    """A LARGE random pool mirroring the revision arm's boundary/mid-sentence
    MIXTURE, with a marker-sized span excised in both cases. Returns the pool
    and the attrition record."""
    pool, attr = [], Counter()
    for model, n_want in per_model_target.items():
        pids = [pid for (m, pid), tr in cache.items() if m == model and eligible(tr)]
        if not pids: continue
        made = att = 0
        cap = n_want * 60
        while made < n_want and att < cap:
            att += 1
            pid = pids[rng.randrange(len(pids))]
            tr = cache[(model, pid)]; fr = tr.get("full_response", "")
            if len(fr) < MIN_LEN:
                attr["too short"] += 1; continue
            lo, hi = 1000, len(fr) - 800
            if hi <= lo:
                attr["no usable range"] += 1; continue
            # MIXTURE: boundary with the revision arm's own probability
            if rng.random() < p_boundary:
                bounds = [x for x in sentence_starts(fr) if lo <= x <= hi]
                if not bounds:
                    attr["no boundary in range"] += 1; continue
                cs = bounds[rng.randrange(len(bounds))]
                kind = "boundary"
            else:
                cs = rng.randrange(lo, hi)
                kind = "midsentence"
            mlen = int(mlens[rng.randrange(len(mlens))])
            ce = min(cs + mlen, len(fr) - 1)
            b, a = extract_window(fr, cs, ce)
            if b is None:
                attr["window failed"] += 1; continue
            ev = pack("random", model, pid, b, a, cs, fr, tr)
            ev["draw_kind"] = kind
            pool.append(ev); made += 1
        if made < n_want:
            attr[f"{model} short by {n_want-made}"] += 1
    return pool, attr

In [ ]:
def _ls(y, X):
    b, *_ = np.linalg.lstsq(X, y, rcond=None); return b

def fit_boot(y, X, groups=None, n_boot=N_BOOT, seed=0):
    beta = _ls(y, X); rng = np.random.RandomState(seed)
    n = len(y); draws = np.empty((n_boot, X.shape[1]))
    if groups is None:
        for b in range(n_boot):
            i = rng.randint(0, n, n); draws[b] = _ls(y[i], X[i])
    else:
        gi = defaultdict(list)
        for i, g in enumerate(groups): gi[g].append(i)
        arrs = [np.array(v) for v in gi.values()]; G = len(arrs)
        for b in range(n_boot):
            i = np.concatenate([arrs[j] for j in rng.randint(0, G, G)])
            draws[b] = _ls(y[i], X[i])
    return beta, draws

def ci_p(col):
    lo, hi = np.percentile(col, [2.5, 97.5])
    p = 2 * min((col <= 0).mean(), (col >= 0).mean())
    return float(lo), float(hi), float(min(1.0, p))


def design(ev, control_repetition=True):
    y = np.array([e["divergence"] for e in ev], float)
    r = np.array([1.0 if e["arm"] == "revision" else 0.0 for e in ev])
    cols = [np.ones_like(y), r]
    cm = None
    if control_repetition:
        c = np.array([e["max_jaccard"] for e in ev], float)
        cm = float(c.mean()); cc = c - cm
        cols += [cc, r * cc]
    return y, np.column_stack(cols), [e["trace_id"] for e in ev], cm


def main():
    cache, results = {}, {}
    print(f"Loading {EMBED_MODEL_NAME} ...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)

    rev, mlens = build_revision(cache)
    p_boundary = 1.0 - float(np.mean([e["frag_before"] for e in rev]))
    print(f"  revision {len(rev)} events, {len({e['trace_id'] for e in rev})} traces")
    print(f"  marker length median {np.median(mlens):.0f} chars")
    print(f"  revision boundary rate (1 - frag_before) = {p_boundary:.3f}")
    print(f"  -> the pool will mirror this mixture, not force all boundaries")

    per_model = {m: c * POOL_MULT for m, c in Counter(e["model"] for e in rev).items()}
    print(f"\n  building a pool {POOL_MULT}x the revision count "
          f"({sum(per_model.values())} target) ...")
    pool, attr = build_pool(cache, random.Random(SEED), per_model, mlens, p_boundary)
    print(f"  pool: {len(pool)} events  ({len(pool)/sum(per_model.values()):.0%} of target)")
    if attr:
        print("  attrition:")
        for k, v in attr.most_common(6): print(f"    {v:>6}  {k}")
    results["pool"] = {"n": len(pool), "target": sum(per_model.values()),
                       "p_boundary": p_boundary, "attrition": dict(attr)}

    print(f"\n  embedding {len(rev)+len(pool)} windows (this is the slow part) ...")
    for e in rev + pool:
        B = emb.encode(e["before"], normalize_embeddings=True)
        A = emb.encode(e["after"],  normalize_embeddings=True)
        cb, ca = B.mean(axis=0), A.mean(axis=0)
        e["divergence"] = 1.0 - float(np.dot(cb, ca) /
                                      (np.linalg.norm(cb) * np.linalg.norm(ca)))
        e["max_jaccard"] = max_jaccard(e["before"], e["after"])

    # ---------------- P2: did the mixture match? ---------------------------
    print(f"\n{'#'*74}\n# 1 — DID THE MIXTURE MATCH? (P2)\n{'#'*74}")
    print(f"  {'covariate':<14}{'revision':>11}{'pool':>11}{'gap':>10}{'p':>11}")
    bal, struct_bad = {}, 0
    for k in ("n_before", "n_after", "frag_before", "frag_after", "rel_pos",
              "difficulty", "max_jaccard"):
        a = np.array([e[k] for e in rev], float); b = np.array([e[k] for e in pool], float)
        _, p = mannwhitneyu(a, b, alternative="two-sided")
        bal[k] = {"rev": float(a.mean()), "pool": float(b.mean()),
                  "gap": float(a.mean()-b.mean()), "p": float(p)}
        if k in ("n_before", "n_after", "frag_before", "frag_after") and p < 0.05:
            struct_bad += 1
        print(f"  {k:<14}{a.mean():>11.4f}{b.mean():>11.4f}"
              f"{a.mean()-b.mean():>+10.4f}{p:>10.3g}{'*' if p<0.05 else ' '}")
    results["balance"] = bal
    print(f"\n  structural covariates imbalanced: {struct_bad}/4")
    matched_ok = struct_bad <= 1
    print(f"  -> {'acceptably matched' if matched_ok else 'STILL IMBALANCED — treat the raw estimate as confounded'}")

    # ---------------- P1: draw stability ----------------------------------
    print(f"\n{'#'*74}\n# 2 — DRAW STABILITY (P1, the serious one)\n{'#'*74}")
    print(f"  Refitting on {N_DRAWS} independent revision-sized subsets of the pool.")
    print(f"  Each subset is drawn per-model to match the revision arm's model mix.")
    by_model = defaultdict(list)
    for e in pool: by_model[e["model"]].append(e)
    need = Counter(e["model"] for e in rev)
    rng = np.random.RandomState(0)
    ests = {"raw": [], "rep": []}
    for d in range(N_DRAWS):
        sub = []
        for m, n in need.items():
            avail = by_model[m]
            if not avail: continue
            idx = rng.choice(len(avail), size=min(n, len(avail)), replace=False)
            sub += [avail[i] for i in idx]
        for tag, ctrl in (("raw", False), ("rep", True)):
            y, X, tid, _ = design(rev + sub, control_repetition=ctrl)
            ests[tag].append(float(_ls(y, X)[1]))
    draw = {}
    for tag, lab in (("raw", "no repetition control"),
                     ("rep", "repetition controlled")):
        a = np.array(ests[tag])
        lo, hi = np.percentile(a, [2.5, 97.5])
        frac_pos = float((a > 0).mean())
        draw[tag] = {"mean": float(a.mean()), "sd": float(a.std()),
                     "range": [float(a.min()), float(a.max())],
                     "pct2_5": float(lo), "pct97_5": float(hi),
                     "frac_positive": frac_pos}
        print(f"\n  --- is_revision across draws, {lab} ---")
        print(f"    mean {a.mean():+.4f}   sd {a.std():.4f}")
        print(f"    range [{a.min():+.4f}, {a.max():+.4f}]")
        print(f"    2.5-97.5 pct [{lo:+.4f}, {hi:+.4f}]")
        print(f"    positive in {frac_pos:.0%} of draws")
    results["draw_stability"] = draw

    # ---------------- the pooled estimate on the FULL pool -----------------
    print(f"\n{'#'*74}\n# 3 — ESTIMATE ON THE FULL POOL (draw noise minimised)\n{'#'*74}")
    full = {}
    for tag, ctrl in (("raw", False), ("rep", True)):
        y, X, tid, cm = design(rev + pool, control_repetition=ctrl)
        b, dd = fit_boot(y, X, groups=tid)
        lo1, hi1, p1 = ci_p(dd[:, 1])
        entry = {"is_revision": {"beta": float(b[1]), "ci95": [lo1, hi1], "p": p1}}
        lab = "no repetition control" if tag == "raw" else "repetition controlled"
        print(f"\n  --- {lab}  (n_rev={len(rev)}, n_pool={len(pool)}) ---")
        print(f"    is_revision {b[1]:>+9.4f}  [{lo1:+.4f}, {hi1:+.4f}]  p={fmt_p(p1)}")
        if ctrl:
            lo3, hi3, p3 = ci_p(dd[:, 3])
            entry["interaction"] = {"beta": float(b[3]), "ci95": [lo3, hi3], "p": p3}
            print(f"    interaction {b[3]:>+9.4f}  [{lo3:+.4f}, {hi3:+.4f}]  p={fmt_p(p3)}")
            c = np.array([e["max_jaccard"] for e in rev + pool], float)
            qs = np.quantile(c, np.linspace(0, 1, NQ+1))
            c_low = float(c[c < qs[1]].mean())
            dl = dd[:, 1] + dd[:, 3]*(c_low - cm); est = b[1] + b[3]*(c_low - cm)
            llo, lhi, lp = ci_p(dl)
            entry["low_repetition"] = {"beta": float(est), "ci95": [llo, lhi], "p": lp}
            print(f"    low-repetition simple effect {est:+.4f}  "
                  f"[{llo:+.4f}, {lhi:+.4f}]  p={fmt_p(lp)}")
        full[tag] = entry
    results["full_pool"] = full

    # ---------------- READ, gated properly ---------------------------------
    print(f"\n{'#'*74}\n# READ\n{'#'*74}")
    d_rep = draw["rep"]; f_rep = full["rep"]["is_revision"]
    stable = d_rep["pct2_5"] > 0
    consistent = d_rep["frac_positive"] >= 0.95
    print(f"  across-draw 2.5th pct (repetition controlled): {d_rep['pct2_5']:+.4f}")
    print(f"  positive in {d_rep['frac_positive']:.0%} of {N_DRAWS} draws")
    print(f"  full-pool estimate: {f_rep['beta']:+.4f} "
          f"[{f_rep['ci95'][0]:+.4f}, {f_rep['ci95'][1]:+.4f}]")
    print()
    if not matched_ok:
        print("  MATCHING STILL FAILED. Whatever follows is confounded by window")
        print("  structure; the arms cannot be equated by construction either. Report")
        print("  the repetition-controlled estimate WITH the structural imbalance")
        print("  stated as a limitation, and do not lead with divergence.")
        print()
    if stable and consistent:
        print("  STABLE ACROSS DRAWS. The estimate stays positive in essentially every")
        print("  independent draw of the baseline, so it is not an artifact of one")
        print("  lucky set of random positions. Report the FULL-POOL estimate with")
        print("  cluster-robust CIs, and report the across-draw interval alongside it")
        print("  as the honest uncertainty.")
    elif consistent and not stable:
        print("  DIRECTIONALLY CONSISTENT BUT NOT STABLE. The sign holds in most draws")
        print("  but the across-draw interval includes zero, so the single-run CIs")
        print("  were too narrow. Report the ACROSS-DRAW interval and describe the")
        print("  effect as suggestive rather than established.")
    else:
        print("  NOT STABLE. The estimate changes sign or straddles zero across")
        print("  independent draws of the baseline. The divergence effect is NOT")
        print("  established: the earlier tight CIs reflected one draw, not the")
        print("  underlying uncertainty. Report the null.")
    print(f"\n  Note the earlier runs gave +0.0147 and +0.0093 from two draws of a")
    print(f"  revision-sized baseline. The spread above is why that happened, and")
    print(f"  why a count-matched baseline was the wrong design choice.")

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(results, open(OUT_PATH, "w"), indent=2, default=str)
    print(f"\nWrote {OUT_PATH}")


if __name__ == "__main__":
    main()

---

## How to read it

**Section 2 decides this.** If the across-draw interval straddles zero, the divergence effect is not established — the earlier tight CIs reflected one draw, not the uncertainty. Report the null.

If it stays positive across essentially every draw, report the **full-pool** estimate with cluster-robust CIs, and the **across-draw interval** beside it as the honest uncertainty.

Either way, if section 1 still shows structural imbalance, the arms cannot be equated by construction and that belongs in the limitations — not buried.